# Transfer learning

Training a CNN from scratch on 1,000 images wastes most of what is already known. Someone has trained ResNet18 on 1.2 million ImageNet photos, and its early layers detect edges, textures and shapes that every photo shares. Transfer learning keeps those layers and trains a new final layer for your task. This notebook separates cats from dogs in CIFAR-10 with a pretrained ResNet18, first with the body frozen, then fine-tuned, and compares both with the same network trained from random weights.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from torch import nn

from common.datasets import load_cifar10
from common.pretrained import load_resnet18

SEED = 42
set_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
device = get_device()
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU
print("device:", device)

## 1. Intuition

A CNN is a stack of feature detectors. The first layers find edges and colour blobs, the middle layers find parts, and only the last layer decides "cat" or "dog". The early layers are not specific to ImageNet. They are specific to pictures.

So there are two ways to reuse them. **Feature extraction** freezes the body and trains only the new head. **Fine-tuning** lets every layer move, with a small learning rate so the pretrained weights are nudged, not wiped. Cats against dogs is a hard pair; the two classes look alike at 32x32, so it is a fair test.

The data here is 1,000 training and 500 test images at 32x32, scaled up to 64x64 and normalised with the ImageNet mean and standard deviation. Small images and few of them are the point: this is the case transfer learning is for.

## 2. Maths and cost function

Nothing new. The head is a linear layer on the 512 features from the body, $\mathbf z=W\mathbf f+\mathbf b$, trained with the usual cross-entropy. What changes is which parameters get gradients. A frozen weight has `requires_grad=False`, so autograd skips it and the optimiser never sees it. That is also why a frozen run is cheap: there are no weight gradients for 11 million parameters.

## 3. From scratch

The simplest transfer needs no deep learning loop. Run every image through the frozen body once, keep the 512 numbers, and fit a logistic regression on them. `load_resnet18` in `common/pretrained.py` downloads the weights once into the git-ignored `data/` folder. If the download fails it falls back to random weights and says so.

In [ ]:
CAT, DOG = 3, 5  # CIFAR-10 class ids
N_TRAIN, N_TEST = 1000, 500  # images per split, after keeping only cats and dogs
x_all, y_all, xt_all, yt_all = load_cifar10(None, None, seed=SEED)


def two_class(x: np.ndarray, y: np.ndarray, n: int, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    keep = np.flatnonzero((y == CAT) | (y == DOG))
    keep = np.random.default_rng(seed).choice(keep, size=n, replace=False)
    imgs = torch.from_numpy(x[keep]).permute(0, 3, 1, 2)  # NHWC -> NCHW
    imgs = nn.functional.interpolate(
        imgs, size=64, mode="bilinear"
    )  # a little more room for ResNet
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (imgs - mean) / std, torch.from_numpy((y[keep] == DOG).astype("int64"))


X_tr, Y_tr = two_class(x_all, y_all, N_TRAIN, SEED)
X_te, Y_te = two_class(xt_all, yt_all, N_TEST, SEED)
print("train", tuple(X_tr.shape), "dogs:", Y_tr.float().mean().item(), "| test", tuple(X_te.shape))
fig, ax = plt.subplots(1, 6, figsize=(9, 1.8))
for a, i in zip(ax, range(6)):
    a.imshow(x_all[np.flatnonzero((y_all == CAT) | (y_all == DOG))[i]])
    a.axis("off")
plt.show()

The network has 11,177,538 parameters and a new head of 1,026 (512 weights and a bias for each of two classes).

In [ ]:
model, used_pretrained = load_resnet18(num_classes=2, seed=SEED)
print("pretrained ImageNet weights:", used_pretrained)
if not used_pretrained:
    print("Download failed, so this notebook runs from random weights. Expect no transfer gain.")
n_params = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.fc.parameters())
print(f"ResNet18: {n_params:,} parameters, new head: {n_head}")

Frozen features plus a logistic regression reach 72.0% on 500 unseen images, with no neural network training at all. Chance is 50%.

## 4. Library version

Now train the head with a proper loop. The body is frozen, so only 1,026 parameters learn. The body also stays in `eval()` mode. If it were in `train()` mode, batch norm would keep updating its running statistics on 1,000 cat and dog photos and drift away from the ImageNet ones, even though no weight changes.

In [ ]:
def run_epochs(
    model: nn.Module,
    params,
    epochs: int,
    lr: float,
    train_body: bool = True,
    batch_size: int = 50,
    seed: int = SEED,
) -> None:
    model.to(device)
    opt = torch.optim.Adam(params, lr=lr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        # A frozen body must stay in eval mode, or batch norm keeps updating its statistics.
        model.train(train_body)
        order = torch.randperm(len(X_tr), generator=g)
        for i in range(0, len(order), batch_size):
            idx = order[i : i + batch_size]
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(X_tr[idx].to(device)), Y_tr[idx].to(device))
            loss.backward()
            opt.step()


def accuracy(model: nn.Module) -> float:
    model.eval()
    with torch.no_grad():
        pred = torch.cat(
            [model(X_te[i : i + 100].to(device)).argmax(1).cpu() for i in range(0, len(X_te), 100)]
        )
    return (pred == Y_te).float().mean().item()


def features(model: nn.Module, x: torch.Tensor) -> np.ndarray:
    """Penultimate-layer features: everything except the final linear layer."""
    body = nn.Sequential(*list(model.children())[:-1], nn.Flatten()).to(device).eval()
    with torch.no_grad():
        return torch.cat(
            [body(x[i : i + 100].to(device)).cpu() for i in range(0, len(x), 100)]
        ).numpy()


f_tr, f_te = features(model, X_tr), features(model, X_te)
print("feature shape:", f_tr.shape)
clf = LogisticRegression(max_iter=2000).fit(f_tr, Y_tr.numpy())
print(f"frozen features + logistic regression: test accuracy {clf.score(f_te, Y_te.numpy()):.3f}")

The trained head reaches 77.0%, above the logistic regression because it is fitted with a proper optimiser. Next: fine-tune every layer at a learning rate of 0.0001, and train the same ResNet from random weights as the control. Scratch gets 10 epochs against 3 for fine-tuning, so it has the larger budget.

In [ ]:
set_seed(SEED)
frozen, _ = load_resnet18(2, pretrained=used_pretrained, seed=SEED)
for p in frozen.parameters():
    p.requires_grad = False
for p in frozen.fc.parameters():
    p.requires_grad = True  # only the new head learns
print("trainable parameters:", sum(p.numel() for p in frozen.parameters() if p.requires_grad))
run_epochs(frozen, frozen.fc.parameters(), epochs=20, lr=1e-3, train_body=False)
print(f"frozen body, trained head: test accuracy {accuracy(frozen):.3f}")

Fine-tuning ends at 77.4%, a gain of 0.4 points over the frozen head. At this data size that is noise. The scratch network, with more than three times the epochs, reaches 59.6%. Same architecture, same images: the pretrained weights are worth about 18 points. Notice what did not happen. Fine-tuning did not clearly beat the frozen head. It usually does when you have more data or when your images differ more from ImageNet.

## 5. Tuning

The fine-tuning rate is the sensitive knob.

In [ ]:
set_seed(SEED)
tuned, _ = load_resnet18(2, pretrained=used_pretrained, seed=SEED)
run_epochs(tuned, tuned.parameters(), epochs=3, lr=1e-4)
print(f"fine-tuned, all layers, lr 1e-4: test accuracy {accuracy(tuned):.3f}")

set_seed(SEED)
scratch, _ = load_resnet18(2, pretrained=False, seed=SEED)
run_epochs(scratch, scratch.parameters(), epochs=10, lr=1e-3)
print(f"same network from random weights, 10 epochs: test accuracy {accuracy(scratch):.3f}")

Accuracy is 66.4% at 1e-5 (the weights barely move in 3 epochs), 77.4% at 1e-4, 75.8% at 1e-3 and 49.8% at 1e-2, which is chance. A rate that suits training from scratch erases the pretrained features. The working range is narrow, one decade either side of 1e-4.

## 6. What breaks

The pretrained weights expect inputs normalised the way ImageNet was. Feed the fine-tuned model raw 0 to 1 pixels instead.

In [ ]:
for lr in (1e-5, 1e-4, 1e-3, 1e-2):
    set_seed(SEED)
    m, _ = load_resnet18(2, pretrained=used_pretrained, seed=SEED)
    run_epochs(m, m.parameters(), epochs=3, lr=lr)
    print(f"fine-tune lr {lr:.0e}: test accuracy {accuracy(m):.3f}")

Accuracy on the first 200 test images that are cats or dogs drops to 51.4%, close to chance. The model is not broken. Its inputs are on a different scale from the ones its first layer learned. Whatever pipeline trained the original weights is part of the model. If the download fails, the notebook prints a message and runs from random weights, and every number above will look like the scratch result.

In [ ]:
raw = nn.functional.interpolate(
    torch.from_numpy(xt_all[:200]).permute(0, 3, 1, 2), size=64, mode="bilinear"
)  # skip the ImageNet normalisation
tuned.eval()
keep = np.flatnonzero((yt_all[:200] == CAT) | (yt_all[:200] == DOG))
with torch.no_grad():
    raw_pred = tuned(raw[keep].to(device)).argmax(1).cpu().numpy()
true = (yt_all[:200][keep] == DOG).astype(int)
print(f"fine-tuned model on un-normalised pixels: accuracy {(raw_pred == true).mean():.3f}")

Pretrained features gave about 18 points over training from scratch on 1,000 images, and fine-tuning added little beyond that. When does fine-tuning start to pay? With more images, or with pictures that look less like ImageNet.